# Alpha Search — Real Data Backtesting Pipeline

**Version:** 0.2.2  
**Colab-ready:** Yes  
**Data:** Real OHLCV only (yfinance)

---

This notebook runs the full Alpha Search backtesting pipeline on **real market data**. It covers:

1. Install dependencies
2. Clone and install Alpha Search
3. Imports
4. Research disclaimer
5. Data source configuration
6. Real data ingestion
7. Data validation
8. Signal generation
9. Backtest engine
10. Transaction costs & slippage
11. Metrics calculation
12. Visualisation
13. CSV export
14. Markdown report export
15. Research conclusion


## 1. Install Dependencies

In [ ]:
# Run this cell first — it installs all required packages.
# On a fresh Colab runtime this takes ~60 seconds.
import sys

!{sys.executable} -m pip install -q \
    yfinance>=0.2.48 \
    pandas>=2.0.0 \
    numpy>=1.26.0 \
    matplotlib>=3.8.0 \
    scipy>=1.12.0 \
    statsmodels>=0.14.0 \
    pydantic>=2.6.0

print("Dependencies installed.")

## 2. Clone and Install Alpha Search

In [ ]:
import os

# Skip cloning if already installed (e.g., running locally)
try:
    import alpha_search
    print(f"alpha_search already installed (version {getattr(alpha_search, '__version__', 'unknown')})")
except ImportError:
    # Clone from GitHub if running on Colab / fresh environment
    if not os.path.exists("alpha-search"):
        !git clone https://github.com/alpha-search/alpha-search.git
    %cd alpha-search
    !pip install -q -e .
    print("alpha_search installed from source.")

## 3. Imports

In [ ]:
from __future__ import annotations

import json
import logging
import os
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd

# Alpha Search modules
from alpha_search.backtest.engine import BacktestEngine
from alpha_search.backtest.costs import CostModel
from alpha_search.research.real_data_pipeline import (
    UNIVERSES,
    UNIVERSE_US_LARGE_CAP,
    UNIVERSE_INDIA_EQUITY,
    UNIVERSE_CRYPTO,
    calculate_metrics,
    export_research_outputs,
    fetch_yfinance_ohlcv,
    generate_breakout_signal,
    generate_mean_reversion_signal,
    generate_momentum_signal,
    load_csv_ohlcv,
    run_real_data_research,
    run_vectorized_backtest,
    validate_ohlcv,
)

# Suppress minor warnings for cleaner notebook output
warnings.filterwarnings("ignore", category=FutureWarning)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)

matplotlib.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.4f}".format)

print("Imports complete.")

## 4. Research Disclaimer

> **⚠ IMPORTANT — READ BEFORE PROCEEDING**
>
> This notebook is for **research and educational purposes only**.  
> It does **not** constitute investment advice.  
> **Past performance does not guarantee future results.**
>
> Backtests are run on historical data and do **not** account for:
> - Survivorship bias (only symbols that existed for the full period are tested)
> - Look-ahead bias (the pipeline uses `shift(1)` throughout, but edge cases may exist)
> - Market impact (large orders cannot be filled at the modelled price)
> - Tax treatment, regulatory restrictions, or account minimums
>
> If a Sharpe ratio is negative, it is reported **honestly** — no results are fabricated.
> If a strategy generates no trades, the notebook will say so clearly.
>
> Always consult a licensed financial advisor before making investment decisions.

## 5. Data Source Configuration

Edit the cell below to select your universe, time period, and cost assumptions.
The default runs the US Large-Cap universe over the last 2 years.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# Universe: 'us_large_cap' | 'india_equity' | 'crypto' | 'all'
# Or supply a custom list:  SYMBOLS = ["AAPL", "MSFT", "TSLA"]
UNIVERSE_NAME = "us_large_cap"
SYMBOLS = UNIVERSES[UNIVERSE_NAME]  # override with a custom list if desired

# yfinance period: '1y' | '2y' | '5y' | 'max'
PERIOD = "2y"

# Bar interval: '1d' | '1h' | '30m'
INTERVAL = "1d"

# Transaction costs in basis points (10 bps = 0.10% per trade)
TRANSACTION_COST_BPS = 10.0   # commission
SLIPPAGE_BPS          = 10.0   # market impact / slippage

# Initial portfolio capital per strategy-symbol combination
INITIAL_CAPITAL = 100_000.0

# Output directory for results
OUTPUT_DIR = "outputs/research_runs"

# CSV fallback: set to a filepath if you want to load local OHLCV instead of yfinance
CSV_FALLBACK_PATH = None  # e.g. "/content/my_data.csv"

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Universe      : {UNIVERSE_NAME} ({len(SYMBOLS)} symbols)")
print(f"Symbols       : {SYMBOLS}")
print(f"Period        : {PERIOD}")
print(f"Interval      : {INTERVAL}")
print(f"Cost (round-trip): {TRANSACTION_COST_BPS + SLIPPAGE_BPS:.0f} bps ({(TRANSACTION_COST_BPS + SLIPPAGE_BPS)/100:.2f}%)")
print(f"Initial capital  : ${INITIAL_CAPITAL:,.0f}")
print(f"Output dir    : {OUTPUT_DIR}")

## 6. Real Data Ingestion

Downloads OHLCV data from Yahoo Finance (via `yfinance`).
Each symbol is fetched independently — if a symbol fails, it is skipped and logged.
**No data is fabricated if a download fails.**

In [ ]:
if CSV_FALLBACK_PATH:
    print(f"Loading OHLCV from CSV: {CSV_FALLBACK_PATH}")
    frames = load_csv_ohlcv(CSV_FALLBACK_PATH)
    succeeded = list(frames.keys())
    failed = [s for s in SYMBOLS if s not in frames]
else:
    print(f"Fetching {len(SYMBOLS)} symbols from Yahoo Finance (period={PERIOD}, interval={INTERVAL})...")
    frames, succeeded, failed = fetch_yfinance_ohlcv(SYMBOLS, period=PERIOD, interval=INTERVAL)

print(f"\nSucceeded : {len(succeeded)} — {succeeded}")
if failed:
    print(f"Failed    : {len(failed)} — {failed} (skipped — no data fabricated)")

if not frames:
    raise RuntimeError(
        "No symbols loaded. Check your internet connection or try a different universe / period."
    )

# Show a sample
sample_sym = succeeded[0]
print(f"\nSample: {sample_sym} ({len(frames[sample_sym])} bars)")
frames[sample_sym].tail(3)

## 7. Data Validation

Each symbol is validated for:
- Required OHLCV columns
- Minimum row count (≥ 20)
- Non-positive close prices
- OHLCV integrity (High ≥ max(Open, Close), Low ≤ min(Open, Close))
- NaN rate
- Large single-day price gaps

In [ ]:
validation_report = {}
valid_frames = {}

for sym, df in frames.items():
    ok, issues = validate_ohlcv(df, sym)
    validation_report[sym] = {"valid": ok, "issues": issues}
    if ok:
        valid_frames[sym] = df
    else:
        print(f"  ✗ {sym} INVALID — {'; '.join(issues[:3])}")

print(f"\nValid symbols   : {len(valid_frames)} / {len(frames)}")
print(f"Invalid symbols : {len(frames) - len(valid_frames)}")

if not valid_frames:
    raise RuntimeError("No valid symbols after data validation. Cannot continue.")

# Compact validation summary table
rows = []
for sym, rep in validation_report.items():
    n_bars = len(frames.get(sym, []))
    rows.append({
        "symbol": sym,
        "valid": rep["valid"],
        "n_bars": n_bars,
        "issues": len(rep["issues"]),
        "first_issue": rep["issues"][0][:60] if rep["issues"] else "",
    })
pd.DataFrame(rows).set_index("symbol")

## 8. Signal Generation

Three strategies are implemented:

| Strategy | Entry condition | Exit condition |
|---|---|---|
| **Momentum** | 20-day rolling return > 0 **and** MA(20) > MA(50) | Rolling return ≤ 0 or MA crossover reverses |
| **Mean Reversion** | z-score(close, 20) < −2.0 | z-score > 0 |
| **Breakout** | Close > Donchian 20-bar high (shifted 1 bar) | Close < Donchian 20-bar low |

All signals use `shift(1)` before rolling to prevent look-ahead bias.

In [ ]:
# Generate signals for each valid symbol
momentum_signals       = {}
mean_reversion_signals = {}
breakout_signals       = {}

for sym, df in valid_frames.items():
    close = df["Close"]
    high  = df.get("High")
    low   = df.get("Low")

    momentum_signals[sym]       = generate_momentum_signal(close, lookback=20, ma_confirm=True)
    mean_reversion_signals[sym] = generate_mean_reversion_signal(close, window=20, z_threshold=2.0)
    breakout_signals[sym]       = generate_breakout_signal(close, high=high, low=low, window=20)

# Display signal statistics for the first symbol
sym0 = list(valid_frames.keys())[0]
print(f"Signal statistics for {sym0}:")
sig_df = pd.DataFrame({
    "momentum":       momentum_signals[sym0],
    "mean_reversion": mean_reversion_signals[sym0],
    "breakout":       breakout_signals[sym0],
})
print(sig_df.describe().T[["mean", "min", "max", "count"]].rename(columns={"count": "n_obs"}))
print(f"\nDays in market (non-zero signal):")
print(sig_df.astype(bool).sum().rename("days_in_market"))

## 9. Backtest Engine

The `BacktestEngine` applies a 1-bar position lag (signals from day *t* are executed at the open of day *t+1*), avoiding unrealistic same-bar fills.

In [ ]:
cost_model = CostModel(
    commission=TRANSACTION_COST_BPS / 10_000.0,
    slippage=SLIPPAGE_BPS / 10_000.0,
)

# Run backtests for all strategies × all symbols
backtest_results = {strat: {} for strat in ("momentum", "mean_reversion", "breakout")}

for sym, df in valid_frames.items():
    close = df["Close"]
    high  = df.get("High")
    low   = df.get("Low")

    for strat, signals in [
        ("momentum",       momentum_signals),
        ("mean_reversion", mean_reversion_signals),
        ("breakout",       breakout_signals),
    ]:
        result = run_vectorized_backtest(
            close=close,
            signal=signals[sym],
            high=high,
            low=low,
            initial_capital=INITIAL_CAPITAL,
            transaction_cost_bps=TRANSACTION_COST_BPS,
            slippage_bps=SLIPPAGE_BPS,
        )
        backtest_results[strat][sym] = result

print(f"Backtests complete: {len(valid_frames)} symbols × 3 strategies = {len(valid_frames)*3} runs")

# Sample trade log
sample_result = backtest_results["momentum"][sym0]
print(f"\nSample trade log ({sym0} / momentum strategy):")
if sample_result.trades:
    trades_df = pd.DataFrame(sample_result.trades)
    print(trades_df.head(10).to_string(index=False))
else:
    print("  No trades generated. Consider relaxing signal thresholds.")

## 10. Transaction Costs & Slippage

This section shows the cost impact for each strategy — the difference in final equity with and without costs.

In [ ]:
print(f"Cost assumptions: {TRANSACTION_COST_BPS} bps commission + {SLIPPAGE_BPS} bps slippage")
print(f"Round-trip cost : {TRANSACTION_COST_BPS + SLIPPAGE_BPS} bps ({(TRANSACTION_COST_BPS + SLIPPAGE_BPS)/100:.2f}%)")
print()

# Compare gross vs net equity for the first valid symbol, momentum strategy
gross = run_vectorized_backtest(
    close=valid_frames[sym0]["Close"],
    signal=momentum_signals[sym0],
    initial_capital=INITIAL_CAPITAL,
    transaction_cost_bps=0,
    slippage_bps=0,
)
net = backtest_results["momentum"][sym0]

gross_final = gross.equity_curve.iloc[-1]
net_final   = net.equity_curve.iloc[-1]
cost_drag   = gross_final - net_final

print(f"{sym0} momentum strategy cost analysis:")
print(f"  Gross final equity : ${gross_final:,.2f}")
print(f"  Net final equity   : ${net_final:,.2f}")
print(f"  Cost drag          : ${cost_drag:,.2f} ({cost_drag/INITIAL_CAPITAL*100:.2f}% of capital)")
print(f"  Number of trades   : {len(net.trades)}")

if len(net.trades) > 0:
    cost_per_trade = cost_drag / len(net.trades)
    print(f"  Avg cost per trade : ${cost_per_trade:,.2f}")

## 11. Metrics Calculation

Key metrics computed per strategy × symbol:

| Metric | Description |
|---|---|
| `total_return` | Cumulative return over full period |
| `annualized_return` | Compound annual growth rate |
| `sharpe_ratio` | Risk-adjusted return (252-day annualised) |
| `sortino_ratio` | Downside-only risk variant of Sharpe |
| `max_drawdown` | Worst peak-to-trough drawdown (negative convention) |
| `calmar_ratio` | Annualised return / abs(max drawdown) |
| `exposure` | Fraction of days with a non-zero position |
| `num_trades` | Total number of round-trip trades |

In [ ]:
metrics_tables = {}

for strat in ("momentum", "mean_reversion", "breakout"):
    rows = []
    for sym in valid_frames:
        m = calculate_metrics(backtest_results[strat][sym])
        m["symbol"] = sym
        rows.append(m)
    df_m = pd.DataFrame(rows).set_index("symbol")
    metrics_tables[strat] = df_m

# Display momentum metrics
METRIC_COLS = ["total_return", "annualized_return", "sharpe_ratio",
               "max_drawdown", "calmar_ratio", "exposure", "num_trades"]

print("=" * 70)
for strat in ("momentum", "mean_reversion", "breakout"):
    df_m = metrics_tables[strat]
    cols = [c for c in METRIC_COLS if c in df_m.columns]
    avg_sharpe = df_m["sharpe_ratio"].mean() if "sharpe_ratio" in df_m.columns else float("nan")
    print(f"\n── {strat.upper()} (avg Sharpe: {avg_sharpe:.3f}) ──")
    display(df_m[cols].style.format({
        "total_return":      "{:.2%}",
        "annualized_return": "{:.2%}",
        "sharpe_ratio":      "{:.3f}",
        "max_drawdown":      "{:.2%}",
        "calmar_ratio":      "{:.3f}",
        "exposure":          "{:.2%}",
        "num_trades":        "{:.0f}",
    }).highlight_max(subset=["sharpe_ratio"], color="#c6efce").highlight_min(subset=["max_drawdown"], color="#ffc7ce"))
print("=" * 70)

## 12. Visualisation

Equity curves, drawdowns, and rolling Sharpe for each strategy.

In [ ]:
def plot_equity_curves(strat_name: str, results_dict: dict, initial_capital: float) -> None:
    """Plot equity curves for all symbols under one strategy."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    fig.suptitle(f"{strat_name.replace('_', ' ').title()} Strategy", fontsize=13)

    # --- Equity curves ---
    ax = axes[0]
    for sym, res in results_dict.items():
        equity = res.equity_curve / initial_capital  # normalise to 1.0
        equity.plot(ax=ax, label=sym, alpha=0.8, linewidth=1.2)
    ax.axhline(1.0, color="grey", linewidth=0.8, linestyle="--")
    ax.set_title("Equity Curve (normalised)")
    ax.set_ylabel("Portfolio value (start = 1.0)")
    ax.legend(fontsize=7, ncol=2)

    # --- Drawdown curves ---
    ax = axes[1]
    for sym, res in results_dict.items():
        equity = res.equity_curve
        rolling_max = equity.cummax()
        drawdown = (equity - rolling_max) / rolling_max
        drawdown.plot(ax=ax, label=sym, alpha=0.7, linewidth=1.1)
    ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
    ax.set_title("Drawdown")
    ax.set_ylabel("Drawdown (%)")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
    ax.legend(fontsize=7, ncol=2)

    # --- Rolling 60-day Sharpe (first symbol only) ---
    ax = axes[2]
    sym0 = list(results_dict.keys())[0]
    eq = results_dict[sym0].equity_curve
    daily_rets = eq.pct_change().dropna()
    roll_sharpe = daily_rets.rolling(60).mean() / (daily_rets.rolling(60).std() + 1e-9) * np.sqrt(252)
    roll_sharpe.plot(ax=ax, color="steelblue", linewidth=1.2)
    ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
    ax.axhline(1, color="green", linewidth=0.8, linestyle=":")
    ax.set_title(f"Rolling 60-day Sharpe ({sym0})")
    ax.set_ylabel("Sharpe ratio")

    plt.tight_layout()
    plt.show()


for strat in ("momentum", "mean_reversion", "breakout"):
    plot_equity_curves(strat, backtest_results[strat], INITIAL_CAPITAL)

In [ ]:
# Cross-strategy Sharpe comparison bar chart
sharpe_data = {}
for strat in ("momentum", "mean_reversion", "breakout"):
    sharpe_data[strat] = metrics_tables[strat]["sharpe_ratio"]

sharpe_df = pd.DataFrame(sharpe_data)

fig, ax = plt.subplots(figsize=(max(8, len(valid_frames) * 1.2), 4))
sharpe_df.plot(kind="bar", ax=ax, edgecolor="white", width=0.7)
ax.axhline(0, color="black", linewidth=0.8)
ax.axhline(1, color="green", linewidth=0.8, linestyle=":", label="Sharpe = 1")
ax.set_title("Sharpe Ratio by Strategy and Symbol")
ax.set_ylabel("Annualised Sharpe Ratio")
ax.set_xlabel("Symbol")
ax.tick_params(axis="x", rotation=30)
ax.legend()
plt.tight_layout()
plt.show()

print("\nCross-strategy Sharpe summary:")
print(sharpe_df.describe().T[["mean", "min", "max", "std"]].round(3))

## 13. CSV Export

Export metrics and trade logs to CSV files in the output directory.

In [ ]:
import os
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
run_dir = os.path.join(OUTPUT_DIR, timestamp)
os.makedirs(run_dir, exist_ok=True)

# Write per-strategy metrics CSVs
for strat, df_m in metrics_tables.items():
    path = os.path.join(run_dir, f"metrics_{strat}.csv")
    df_m.to_csv(path)
    print(f"  Saved: {path}")

# Write combined metrics summary
summary_rows = []
for strat, df_m in metrics_tables.items():
    tmp = df_m.copy()
    tmp.insert(0, "strategy", strat)
    summary_rows.append(tmp.reset_index())
summary_df = pd.concat(summary_rows, ignore_index=True)
summary_path = os.path.join(run_dir, "strategy_results_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"  Saved: {summary_path}")

# Write trade log
trade_rows = []
for strat in ("momentum", "mean_reversion", "breakout"):
    for sym, res in backtest_results[strat].items():
        for t in res.trades:
            row = dict(t)
            row["strategy"] = strat
            row["symbol"] = sym
            trade_rows.append(row)

if trade_rows:
    trades_path = os.path.join(run_dir, "trade_log.csv")
    pd.DataFrame(trade_rows).to_csv(trades_path, index=False)
    print(f"  Saved: {trades_path} ({len(trade_rows)} trade entries)")
else:
    print("  No trades generated — trade_log.csv not written.")

print(f"\nAll CSVs written to: {run_dir}")

## 14. Markdown Report Export

Generates a `report.md` file with a summary table, honest Sharpe reporting, and methodology notes.

In [ ]:
def write_markdown_report(path: str) -> None:
    run_ts = datetime.now(timezone.utc).isoformat()
    lines = [
        f"# Alpha Search — Backtest Research Report",
        f"",
        f"**Generated:** {run_ts}  ",
        f"**Universe:** {UNIVERSE_NAME} ({len(SYMBOLS)} symbols requested, {len(valid_frames)} valid)  ",
        f"**Period:** {PERIOD} | **Interval:** {INTERVAL}  ",
        f"**Costs:** {TRANSACTION_COST_BPS} bps commission + {SLIPPAGE_BPS} bps slippage  ",
        f"**Initial Capital:** ${INITIAL_CAPITAL:,.0f} per strategy-symbol",
        f"",
        f"---",
        f"",
        f"> **DISCLAIMER:** Research / educational purposes only. "
        f"NOT investment advice. Past performance does not guarantee future results.",
        f"",
        f"---",
    ]

    if failed:
        lines += [
            f"",
            f"## Data Failures",
            f"",
            f"The following symbols could not be fetched and were **skipped** (no data fabricated):",
            f"",
            f"`{', '.join(failed)}`",
        ]

    for strat in ("momentum", "mean_reversion", "breakout"):
        df_m = metrics_tables[strat]
        sharpe_vals = df_m["sharpe_ratio"].dropna()
        avg_sharpe = sharpe_vals.mean() if len(sharpe_vals) > 0 else float("nan")

        if np.isnan(avg_sharpe):
            verdict = "no_results"
        elif avg_sharpe > 1.0:
            verdict = "promising"
        elif avg_sharpe > 0.0:
            verdict = "marginal"
        else:
            verdict = "unprofitable"

        lines += [
            f"",
            f"## {strat.replace('_', ' ').title()} Strategy — Verdict: `{verdict}`",
            f"",
            f"Average Sharpe ratio: **{avg_sharpe:.3f}**  ",
            f"(Negative Sharpe means the strategy lost money on average after costs.)",
            f"",
            f"| Symbol | Total Return | Ann. Return | Sharpe | Max DD | Trades |",
            f"|--------|-------------|-------------|--------|--------|--------|",
        ]
        for sym, row in df_m.iterrows():
            lines.append(
                f"| {sym} "
                f"| {row.get('total_return', float('nan')):.2%} "
                f"| {row.get('annualized_return', float('nan')):.2%} "
                f"| {row.get('sharpe_ratio', float('nan')):.3f} "
                f"| {row.get('max_drawdown', float('nan')):.2%} "
                f"| {int(row.get('num_trades', 0))} |"
            )

    lines += [
        f"",
        f"---",
        f"",
        f"## Methodology Notes",
        f"",
        f"- **1-bar position lag**: All signals are shifted by 1 bar before execution to prevent look-ahead bias.",
        f"- **Donchian channels** use `shift(1)` before `rolling()` to ensure the channel is computed on prior bars only.",
        f"- **Transaction costs** deduct commission and slippage on every position change.",
        f"- **Negative max_drawdown** follows the convention that drawdown is reported as a negative fraction.",
        f"- **No survivorship bias correction** — only symbols that existed for the full period are included.",
    ]

    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")


report_path = os.path.join(run_dir, "report.md")
write_markdown_report(report_path)
print(f"Report written to: {report_path}")

# Show the first 50 lines
with open(report_path) as f:
    for i, line in enumerate(f):
        if i >= 50:
            print("... (truncated)")
            break
        print(line, end="")

## 15. Research Conclusion

This section prints a final verdict and suggests next steps.

In [ ]:
print("=" * 65)
print("  RESEARCH CONCLUSION")
print("=" * 65)
print(f"  Universe       : {UNIVERSE_NAME}")
print(f"  Valid symbols  : {len(valid_frames)} / {len(SYMBOLS)}")
if failed:
    print(f"  Failed symbols : {failed}")
print()

_VERDICT_DESC = {
    "promising":    "Sharpe > 1.0 — worth further investigation",
    "marginal":     "0 < Sharpe ≤ 1.0 — positive but weak edge",
    "unprofitable": "Sharpe ≤ 0 — strategy lost money after costs",
    "no_results":   "No trades generated — check signal parameters",
}

for strat in ("momentum", "mean_reversion", "breakout"):
    df_m = metrics_tables[strat]
    sharpe_vals = df_m["sharpe_ratio"].dropna()
    avg_sharpe = sharpe_vals.mean() if len(sharpe_vals) > 0 else float("nan")

    if np.isnan(avg_sharpe):
        verdict = "no_results"
    elif avg_sharpe > 1.0:
        verdict = "promising"
    elif avg_sharpe > 0.0:
        verdict = "marginal"
    else:
        verdict = "unprofitable"

    desc = _VERDICT_DESC[verdict]
    sharpe_str = f"{avg_sharpe:.3f}" if not np.isnan(avg_sharpe) else "n/a"
    print(f"  {strat:<18} {verdict:<14} avg_sharpe={sharpe_str}")
    print(f"    → {desc}")
    print()

print()
print("  Suggested next steps:")
print("  1. Walk-forward validation — split into in-sample / out-of-sample periods.")
print("  2. Parameter sensitivity — test lookback windows (10, 20, 40, 60 bars).")
print("  3. Regime filtering — overlay a VIX or trend-strength filter.")
print("  4. Portfolio construction — equal-weight, vol-scaled, or MVO allocation.")
print("  5. Survivorship bias check — test on a delisted + current universe.")
print()
print("  ⚠  REMINDER: This is research only. Not investment advice.")
print(f"  Output dir: {run_dir}")
print("=" * 65)

---

## Bonus: Run the Full Automated Pipeline

The `run_real_data_research()` orchestrator does everything above in one call
and also saves figures automatically.

In [ ]:
# Uncomment to run the full automated pipeline:

# results = run_real_data_research(
#     universe=UNIVERSE_NAME,
#     period=PERIOD,
#     interval=INTERVAL,
#     output_dir=OUTPUT_DIR,
#     transaction_cost_bps=TRANSACTION_COST_BPS,
#     slippage_bps=SLIPPAGE_BPS,
# )
#
# print("Momentum verdict  :", results["momentum"]["verdict"])
# print("Mean-Rev verdict  :", results["mean_reversion"]["verdict"])
# print("Breakout verdict  :", results["breakout"]["verdict"])
# print("Output dir        :", results["output_dir"])

print("Full pipeline is commented out to avoid duplicate yfinance downloads.")
print("Uncomment the cell above to run it.")